# 📊 Performance Pipeline Comparison
**Dataset:** mudah.my Malaysia Property Listings

This notebook benchmarks **4 data processing pipelines** on the same dataset.

| Pipeline | Method |
|----------|--------|
| 1 | Pandas Baseline (unoptimized) |
| 2 | Pandas Optimized (dtype, usecols, categorical, vectorized) |
| 3 | Polars |
| 4 | DuckDB |

**Same task across all methods:**
- Drop missing values
- Remove duplicates
- Group by `state`
- Calculate count and mean price per state

**Metrics measured:** time (sec), CPU (%), memory (MB), throughput (records/sec)

## 📦 Step 1 — Install & Import
Install and import the relevant library for all the optimization method used in the data processing pipelines.

In [1]:
!pip install -q polars duckdb psutil

import pandas as pd
import polars as pl
import duckdb
import psutil
import os
import time
import gc
import psutil, os
process = psutil.Process(os.getpid())
from datetime import datetime

## ⚙️ Step 2 — File Configuration
Set up to read the properties listings dataset extracted from mudah.my and save the performance before and after optimization in csv files.

In [11]:
DATA = 'mudah_goe.csv'

# Output files
PERFORMANCE_BEFORE = 'performance_before.csv'
PERFORMANCE_AFTER  = 'performance_after.csv'

# Columns used in processing
COLS_NEEDED = ['listing_id', 'property_type', 'location', 'state',
               'price_rm', 'bedrooms', 'bathrooms', 'size_sqft', 'tenure']

# Processing task — group by this column
GROUP_COL   = 'state'
NUMERIC_COL = 'price_rm'

# Verify file exists
if os.path.exists(DATA):
    size_mb = os.path.getsize(DATA) / (1024 * 1024)
    print(f'✅ Found {DATA} ({size_mb:.1f} MB)')
else:
    print(f'❌ {DATA} not found — upload your combined CSV first')

✅ Found mudah_goe.csv (37.3 MB)


## 🔧 Step 3 — Helper: Metrics Collection
The function set up the function to collect the benchmark metric, including:

*   Memory Usage (MB)
*   CPU Usage (%)
*   Total Processing Time (second)
*   Throughput (Records/second)

In [12]:
all_run_records = []   # stores every individual run result

def run_benchmark(method_name, fn, total_records, runs=3):
    print(f'\n  Running: {method_name} ({runs} runs)...')

    all_times, all_cpu, all_mem, all_tput = [], [], [], []
    result = None

    for run in range(1, runs + 1):
        gc.collect()
        time.sleep(2)

        # CPU: measure over the fn() window using process-level tracking
        process.cpu_percent(interval=None)

        # Memory: snapshot before
        mem_before = get_memory_mb()
        time_start = time.perf_counter()

        result = fn()

        time_end    = time.perf_counter()
        elapsed     = time_end - time_start

        # CPU: measure over actual elapsed window, capped at 100%
        cpu_percent = min(process.cpu_percent(interval=None), 100.0)

        # Memory: take peak during the run instead of delta
        mem_after   = get_memory_mb()
        memory_used = max(mem_after - mem_before, get_memory_mb() - mem_before, 0)

        throughput  = total_records / elapsed if elapsed > 0 else 0

        all_times.append(elapsed)
        all_cpu.append(cpu_percent)
        all_mem.append(memory_used)
        all_tput.append(throughput)

        # store individual run
        all_run_records.append({
            'method':                     method_name,
            'run':                        run,
            'time_sec':                   round(elapsed, 4),
            'cpu_percent':                round(cpu_percent, 2),
            'memory_mb':                  round(memory_used, 2),
            'throughput_records_per_sec': round(throughput, 0),
        })

        print(f'    Run {run}/{runs} → {elapsed:.3f}s | '
              f'CPU: {cpu_percent:.1f}% | '
              f'Mem: {memory_used:.1f}MB | '
              f'Throughput: {throughput:,.0f} rec/s')

        gc.collect()
        time.sleep(1)

    metrics = {
        'method':                     method_name,
        'time_sec':                   round(sum(all_times) / runs, 4),
        'time_min':                   round(min(all_times), 4),
        'time_max':                   round(max(all_times), 4),
        'cpu_percent':                round(sum(all_cpu) / runs, 2),
        'memory_mb':                  round(sum(all_mem) / runs, 2),
        'throughput_records_per_sec': round(sum(all_tput) / runs, 0),
    }

    print(f'    ── Avg: {metrics["time_sec"]}s | '
          f'Min: {metrics["time_min"]}s | '
          f'Max: {metrics["time_max"]}s')

    return metrics, result


def get_memory_mb():
    return process.memory_info().rss / (1024 * 1024)

print('✅ Benchmark helper ready.')

✅ Benchmark helper ready.


## 📋 Step 4 — Get Total Record Count

In [13]:
# Count total rows without loading full data into memory
with open(DATA, 'r', encoding='utf-8-sig') as f:
    total_records = sum(1 for _ in f) - 1  # subtract header

print(f'Total records in dataset : {total_records:,}')

# Quick peek at structure
df_peek = pd.read_csv(DATA, nrows=3)
print(f'Columns                  : {list(df_peek.columns)}')
print(f'\nFirst 3 rows:')
display(df_peek)

Total records in dataset : 922,509
Columns                  : ['listing_id', 'property_type', 'description', 'location', 'state', 'price_rm', 'mortgage_est_rm', 'land_title', 'size_sqft', 'bedrooms', 'bathrooms', 'tenure', 'seller_type', 'seller_name', 'url', 'img_url']

First 3 rows:


,listing_id,property_type,description,location,state,price_rm,mortgage_est_rm,land_title,size_sqft,bedrooms,bathrooms,tenure,seller_type,seller_name,url,img_url
0,114696325,2-storey Terraced Houseforsale,Eco Cascadia Mount Austin 2 Storey House JB\n🔥...,Johor Bahru,Johor,720000,2869.0,Non Bumi Lot,1614.0,5.0,4.0,Freehold,Agent,NaN,https://www.mudah.my/eco-cascadia-mount-austin...,https://cdn.rnudah.com/images/plain/9f2ae5538d...
1,114709669,2-storey Terraced Houseforsale,Parkland Bandar Layangkasa Pasir Gudang Ori Un...,Pasir Gudang,Johor,468000,1865.0,Non Bumi Lot,1400.0,4.0,3.0,Freehold,Unknown,NaN,https://www.mudah.my/parkland-bandar-layangkas...,https://cdn.rnudah.com/images/plain/a4d62990bd...
2,113358275,New 1-storey Terraced Houseforsale,"Rumah Penjawat Awam harga hanya 𝐑𝐌𝟐𝟒𝟎,𝟎𝟎𝟎！🔥\n🔥...",Kluang,Johor,240000,956.0,Non Bumi Lot,1430.0,3.0,2.0,Freehold,Private,NaN,https://www.mudah.my/rumah-penjawat-awam-harga...,https://cdn.rnudah.com/images/plain/2dd37a967f...


---
## 🐼 Step 5 — Pipeline 1: Pandas Baseline (Unoptimized)
No optimizations — plain `read_csv`, default dtypes, standard groupby.

In [14]:
def pandas_baseline():
    """
    UNOPTIMIZED pandas pipeline — no optimizations at all.
    Data is already clean so no dropna or dedup needed.
    Task: load → groupby state → count + mean price
    """
    df = pd.read_csv(DATA)

    df[NUMERIC_COL] = pd.to_numeric(df[NUMERIC_COL], errors='coerce')

    result = df.groupby(GROUP_COL).agg(
        listing_count=(NUMERIC_COL, 'count'),
        avg_price_rm=(NUMERIC_COL, 'mean')
    ).reset_index()

    result['avg_price_rm'] = result['avg_price_rm'].round(2)
    result = result.sort_values('listing_count', ascending=False).reset_index(drop=True)

    return result


metrics_baseline, result_baseline = run_benchmark(
    'pandas_baseline', pandas_baseline, total_records
)

print(f'\nBaseline result ({len(result_baseline)} states):')
display(result_baseline)

pd.DataFrame([metrics_baseline]).to_csv(PERFORMANCE_BEFORE, index=False)
print(f'\n✅ Saved: {PERFORMANCE_BEFORE}')


  Running: pandas_baseline (3 runs)...


/tmp/ipykernel_4308/2822665002.py:7: DtypeWarning: Columns (0,5,9,10,13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA)


    Run 1/3 → 0.912s | CPU: 98.6% | Mem: 96.9MB | Throughput: 1,011,401 rec/s


/tmp/ipykernel_4308/2822665002.py:7: DtypeWarning: Columns (0,5,9,10,13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA)


    Run 2/3 → 1.124s | CPU: 97.0% | Mem: 15.0MB | Throughput: 821,035 rec/s


/tmp/ipykernel_4308/2822665002.py:7: DtypeWarning: Columns (0,5,9,10,13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA)


    Run 3/3 → 0.834s | CPU: 99.5% | Mem: 1.5MB | Throughput: 1,106,212 rec/s
    ── Avg: 0.9565s | Min: 0.8339s | Max: 1.1236s

Baseline result (8 states):


,state,listing_count,avg_price_rm
0,Johor,10766,1718555.80
1,Negeri Sembilan,10043,1096927.08
2,Kedah,8702,1780977.89
3,Melaka,8126,914871.95
4,Kelantan,1671,967058.03
5,30868.0,0,NaN
6,2104.0,0,NaN
7,2989,0,NaN



✅ Saved: performance_before.csv


---
## ⚡ Step 6 — Pipeline 2: Pandas Optimized
| # | Technique | What it does |
|---|-----------|-----------|
| 1 | usecols | Loads only 13 needed columns instead of all 16. |
| 2 | Explicit dtype | Tells pandas the column types upfront so it skips the inference step on read_csv |
| 3 | pd.Categorical | Converts low-cardinality string columns (state, tenure, etc.) to integer codes. Less RAM, faster groupby |
| 4 | .str.strip().str.title() | Cleans location strings using C-speed vectorised operations instead of a Python loop |

In [15]:
def pandas_optimized():
    """
    OPTIMIZED pandas pipeline.
    Techniques:
    1. usecols        — load only 13 needed columns instead of all 16
    2. Explicit dtype — skip type inference on read_csv
    3. pd.Categorical — low-cardinality strings → integer codes (less RAM, faster groupby)
    4. .str.strip().str.title() — vectorised C-speed string cleaning
    """
    NEEDED_COLS = [
        'listing_id', 'description', 'property_type', 'location', 'state',
        'price_rm', 'mortgage_est_rm', 'land_title',
        'size_sqft', 'bedrooms', 'bathrooms', 'tenure', 'seller_type'
    ]
    existing_cols = [c for c in NEEDED_COLS if c in df_peek.columns]

    # Optimization 1 + 2: usecols + explicit dtype
    dtype_map = {'listing_id': 'str'}
    existing_dtypes = {k: v for k, v in dtype_map.items() if k in existing_cols}

    df = pd.read_csv(
        DATA,
        usecols=existing_cols,
        dtype=existing_dtypes,
        engine='c',
        na_values=['', 'None', 'nan'],
    )

    # Convert numeric cols safely — bad values become NaN
    for col in ['price_rm', 'mortgage_est_rm', 'size_sqft', 'bedrooms', 'bathrooms']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Fill missing — numeric cols use median, text cols use 'Unknown'
    for col in ['bedrooms', 'bathrooms', 'size_sqft', 'mortgage_est_rm']:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].median())
    for col in ['property_type', 'location', 'state', 'tenure',
                'seller_type', 'land_title', 'description']:
        if col in df.columns:
            df[col] = df[col].fillna('Unknown')

    # Optimization 3: pd.Categorical
    for col in ['property_type', 'state', 'tenure', 'seller_type', 'land_title']:
        if col in df.columns:
            df[col] = df[col].astype('category')

    # Optimization 4: vectorised string cleaning
    if 'location' in df.columns:
        df['location'] = df['location'].str.strip().str.title()

    # Same groupby task as baseline for fair comparison
    result = (
        df.groupby(GROUP_COL, observed=True)[NUMERIC_COL]
        .agg(['mean', 'count'])
        .rename(columns={'mean': 'avg_price_rm', 'count': 'listing_count'})
        .reset_index()
        .sort_values('listing_count', ascending=False)
        .reset_index(drop=True)
    )
    result['avg_price_rm'] = result['avg_price_rm'].round(2)

    return result


metrics_optimized, result_optimized = run_benchmark(
    'pandas_optimized', pandas_optimized, total_records
)

print(f'\nOptimized result ({len(result_optimized)} states):')
display(result_optimized)


  Running: pandas_optimized (3 runs)...


/tmp/ipykernel_4308/1729898736.py:21: DtypeWarning: Columns (5,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


    Run 1/3 → 0.847s | CPU: 99.1% | Mem: 2.7MB | Throughput: 1,088,852 rec/s


/tmp/ipykernel_4308/1729898736.py:21: DtypeWarning: Columns (5,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


    Run 2/3 → 0.941s | CPU: 98.8% | Mem: 0.0MB | Throughput: 980,341 rec/s


/tmp/ipykernel_4308/1729898736.py:21: DtypeWarning: Columns (5,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


    Run 3/3 → 0.926s | CPU: 100.0% | Mem: 0.0MB | Throughput: 996,023 rec/s
    ── Avg: 0.9048s | Min: 0.8472s | Max: 0.941s

Optimized result (9 states):


,state,avg_price_rm,listing_count
0,Johor,1718555.80,10766
1,Negeri Sembilan,1096927.08,10043
2,Kedah,1780977.89,8702
3,Melaka,914871.95,8126
4,Kelantan,967058.03,1671
5,2104.0,NaN,0
6,30868.0,NaN,0
7,2989,NaN,0
8,Unknown,NaN,0


---
## 🐻‍❄️ Step 7 — Pipeline 3: Polars
Rust-based DataFrame library — lazy evaluation, multi-threaded, columnar.

In [16]:
def polars_pipeline():
    """
    Polars pipeline — same task as baseline.
    Uses lazy evaluation (scan_csv) for memory efficiency.
    """
    existing_cols = [c for c in COLS_NEEDED if c in df_peek.columns]

    # Use lazy API — doesn't load data until .collect()
    lf = pl.scan_csv(
        DATA,
        infer_schema_length=0,
        null_values=['', 'None', 'nan', 'null'],
        ignore_errors=True,
        schema_overrides={
            'listing_id':      pl.Utf8,
            'price_rm':        pl.Utf8,
            'mortgage_est_rm': pl.Utf8,
            'size_sqft':       pl.Utf8,
            'bedrooms':        pl.Utf8,
            'bathrooms':       pl.Utf8,
        }
    )

    # Select only needed columns that exist
    available = lf.collect_schema().names()
    cols_to_select = [c for c in existing_cols if c in available]
    lf = lf.select(cols_to_select)

    # Safely cast numeric columns after loading
    numeric_cols = ['price_rm', 'mortgage_est_rm', 'size_sqft', 'bedrooms', 'bathrooms']
    cast_exprs = [
        pl.col(c).cast(pl.Float64, strict=False)
        for c in numeric_cols if c in cols_to_select
    ]
    if cast_exprs:
        lf = lf.with_columns(cast_exprs)

    # Drop nulls in key columns
    lf = lf.drop_nulls(subset=[GROUP_COL, NUMERIC_COL])

    # Remove duplicates
    lf = lf.unique(subset=['listing_id'], keep='last')

    # Groupby + aggregation
    lf = lf.group_by(GROUP_COL).agg([
        pl.count(NUMERIC_COL).alias('listing_count'),
        pl.mean(NUMERIC_COL).round(2).alias('avg_price_rm'),
    ]).sort('listing_count', descending=True)

    # Execute
    result_pl = lf.collect()

    # Convert to pandas for consistent output
    return result_pl.to_pandas()


metrics_polars, result_polars = run_benchmark(
    'polars', polars_pipeline, total_records
)

print(f'\nPolars result ({len(result_polars)} states):')
display(result_polars)


  Running: polars (3 runs)...
    Run 1/3 → 0.046s | CPU: 100.0% | Mem: 0.5MB | Throughput: 20,193,163 rec/s
    Run 2/3 → 0.074s | CPU: 93.9% | Mem: 1.8MB | Throughput: 12,443,804 rec/s
    Run 3/3 → 0.046s | CPU: 100.0% | Mem: 0.2MB | Throughput: 20,171,524 rec/s
    ── Avg: 0.0552s | Min: 0.0457s | Max: 0.0741s

Polars result (5 states):


,state,listing_count,avg_price_rm
0,Johor,10766,1718555.80
1,Negeri Sembilan,10043,1096927.08
2,Kedah,8702,1780977.89
3,Melaka,8126,914871.95
4,Kelantan,1671,967058.03


---
## 🦆 Step 8 — Pipeline 4: DuckDB
In-process SQL OLAP engine — reads CSV directly with SQL, columnar vectorized execution.

In [17]:
def duckdb_pipeline():
    """
    DuckDB pipeline — same groupby task using SQL.
    Reads CSV directly without loading into pandas first.
    Uses columnar vectorized execution.
    """
    con = duckdb.connect(database=':memory:')

    query = f"""
        SELECT
            {GROUP_COL},
            COUNT(*)                                         AS listing_count,
            ROUND(AVG(TRY_CAST({NUMERIC_COL} AS DOUBLE)), 2) AS avg_price_rm
        FROM read_csv_auto(
            '{DATA}',
            null_padding  = true,
            ignore_errors = true
        )
        WHERE {GROUP_COL}   IS NOT NULL
          AND {NUMERIC_COL} IS NOT NULL
        GROUP BY {GROUP_COL}
        ORDER BY listing_count DESC
    """

    result = con.execute(query).df()
    con.close()
    return result


metrics_duckdb, result_duckdb = run_benchmark(
    'duckdb', duckdb_pipeline, total_records
)

print(f'\nDuckDB result ({len(result_duckdb)} states):')
display(result_duckdb)


  Running: duckdb (3 runs)...
    Run 1/3 → 0.182s | CPU: 100.0% | Mem: 0.0MB | Throughput: 5,078,132 rec/s
    Run 2/3 → 0.187s | CPU: 100.0% | Mem: 0.3MB | Throughput: 4,934,608 rec/s
    Run 3/3 → 0.166s | CPU: 100.0% | Mem: 0.1MB | Throughput: 5,555,833 rec/s
    ── Avg: 0.1782s | Min: 0.166s | Max: 0.1869s

DuckDB result (11 states):


,state,listing_count,avg_price_rm
0,Johor,10767,1718555.80
1,Negeri Sembilan,10043,1096927.08
2,Kedah,8702,1780977.89
3,Melaka,8126,914871.95
4,Kelantan,1671,967058.03
5,30868.0,1,NaN
6,2989,1,NaN
7,2104.0,1,NaN
8,2742.0,1,NaN
9,2152.0,1,NaN


---
## 📊 Step 9 — Combine Results & Compare

In [18]:
all_metrics = [
    metrics_baseline,
    metrics_optimized,
    metrics_polars,
    metrics_duckdb,
]

df_perf = pd.DataFrame(all_metrics)
baseline_time = df_perf.loc[df_perf['method'] == 'pandas_baseline', 'time_sec'].values[0]
df_perf['speedup_vs_baseline'] = (baseline_time / df_perf['time_sec']).round(2)

# ── Build long table: one row per run ─────────────────────────────────────────
df_runs = pd.DataFrame(all_run_records)

avg_cols = df_perf[['method','time_sec','cpu_percent','memory_mb',
                     'throughput_records_per_sec','speedup_vs_baseline']].rename(columns={
    'time_sec':                   'avg_time_sec',
    'cpu_percent':                'avg_cpu_%',
    'memory_mb':                  'avg_memory_mb',
    'throughput_records_per_sec': 'avg_throughput',
})

df_long = df_runs.merge(avg_cols, on='method', how='left')

df_long = df_long.rename(columns={
    'time_sec':                   'time_sec',
    'cpu_percent':                'cpu_%',
    'memory_mb':                  'memory_mb',
    'throughput_records_per_sec': 'throughput',
})

method_order = ['pandas_baseline', 'pandas_optimized', 'polars', 'duckdb']
df_long['method'] = pd.Categorical(df_long['method'], categories=method_order, ordered=True)
df_long = df_long.sort_values(['method', 'run']).reset_index(drop=True)

df_display = df_long[['method','run',
                       'time_sec','cpu_%','memory_mb','throughput',
                       'avg_time_sec','avg_cpu_%','avg_memory_mb','avg_throughput',
                       'speedup_vs_baseline']].copy()

df_display = df_display.set_index(['method', 'run'])

# ── FIX: blank avg cols on runs 1 & 3 using NaN (renders as empty, highlights still work) ──
avg_metric_cols = ['avg_time_sec','avg_cpu_%','avg_memory_mb','avg_throughput','speedup_vs_baseline']

df_display_clean = df_display.copy().astype(float, errors='ignore')
for col in avg_metric_cols:
    df_display_clean[col] = [
        v if i % 3 == 1 else float('nan')
        for i, v in enumerate(df_display[col])
    ]

print(f"\n{'='*90}")
print(f"  FULL BENCHMARK RESULTS — {total_records:,} records | 3 runs each")
print(f"{'='*90}\n")

styled = df_display_clean.style \
    .format({
        'time_sec':    '{:.4f}s',
        'cpu_%':       '{:.1f}%',
        'memory_mb':   '{:.2f}',
        'throughput':  '{:,.0f}',
    }, na_rep='') \
    .format({
        'avg_time_sec':        lambda v: f'{v:.4f}s'  if pd.notna(v) else '',
        'avg_cpu_%':           lambda v: f'{v:.1f}%'  if pd.notna(v) else '',
        'avg_memory_mb':       lambda v: f'{v:.2f}'   if pd.notna(v) else '',
        'avg_throughput':      lambda v: f'{v:,.0f}'  if pd.notna(v) else '',
        'speedup_vs_baseline': lambda v: f'{v:.2f}x'  if pd.notna(v) else '',
    }) \
    .set_table_styles([
        {'selector': 'th', 'props': [('font-weight','bold'),('background-color','#f0f0f0')]},
        {'selector': 'td', 'props': [('text-align','center')]},
        {'selector': 'tr:nth-child(3n)', 'props': [('border-bottom','2px solid #999')]},
    ]) \
    .apply(lambda col: [
        'background-color: #fff9e6' if i % 3 == 0
        else 'background-color: #ffffff' if i % 3 == 1
        else 'background-color: #f0f9ff'
        for i in range(len(col))
    ], axis=0) \
    .highlight_min(subset=['avg_time_sec'],    color='lightgreen',  axis=None) \
    .highlight_max(subset=['avg_time_sec'],    color='#ffcccc',     axis=None) \
    .highlight_max(subset=['avg_throughput','speedup_vs_baseline'], color='lightgreen', axis=None)

display(styled)


  FULL BENCHMARK RESULTS — 922,509 records | 3 runs each



## 💾 Step 10 — Save Output Files

In [19]:
# ── Save full run-level data so we can reload the exact same table ────────────
df_long.to_csv('benchmark_full_runs.csv', index=False)
print('✅ Saved: benchmark_full_runs.csv')

# ── Reload & display identical table from CSV ─────────────────────────────────
df_verify = pd.read_csv('benchmark_full_runs.csv')
df_verify['method'] = pd.Categorical(df_verify['method'], categories=method_order, ordered=True)
df_verify = df_verify.sort_values(['method', 'run']).reset_index(drop=True)

df_display_v = df_verify[['method','run',
                           'time_sec','cpu_%','memory_mb','throughput',
                           'avg_time_sec','avg_cpu_%','avg_memory_mb','avg_throughput',
                           'speedup_vs_baseline']].copy().set_index(['method','run'])

# Same blanking: avg cols only on run 2
for col in avg_metric_cols:
    df_display_v[col] = [
        v if i % 3 == 1 else float('nan')
        for i, v in enumerate(df_display_v[col])
    ]

print(f"\n{'='*90}")
print(f"  VERIFIED FROM CSV — same table reloaded from benchmark_full_runs.csv")
print(f"{'='*90}\n")

# Reuse the same styled object — just swap the data
styled_v = df_display_v.style \
    .format({'time_sec':'{:.4f}s','cpu_%':'{:.1f}%','memory_mb':'{:.2f}','throughput':'{:,.0f}'}, na_rep='') \
    .format({
        'avg_time_sec':        lambda v: f'{v:.4f}s'  if pd.notna(v) else '',
        'avg_cpu_%':           lambda v: f'{v:.1f}%'  if pd.notna(v) else '',
        'avg_memory_mb':       lambda v: f'{v:.2f}'   if pd.notna(v) else '',
        'avg_throughput':      lambda v: f'{v:,.0f}'  if pd.notna(v) else '',
        'speedup_vs_baseline': lambda v: f'{v:.2f}x'  if pd.notna(v) else '',
    }) \
    .set_table_styles([
        {'selector': 'th', 'props': [('font-weight','bold'),('background-color','#f0f0f0')]},
        {'selector': 'td', 'props': [('text-align','center')]},
        {'selector': 'tr:nth-child(3n)', 'props': [('border-bottom','2px solid #999')]},
    ]) \
    .apply(lambda col: [
        'background-color: #fff9e6' if i % 3 == 0
        else 'background-color: #ffffff' if i % 3 == 1
        else 'background-color: #f0f9ff'
        for i in range(len(col))
    ], axis=0) \
    .highlight_min(subset=['avg_time_sec'],    color='lightgreen', axis=None) \
    .highlight_max(subset=['avg_time_sec'],    color='#ffcccc',    axis=None) \
    .highlight_max(subset=['avg_throughput','speedup_vs_baseline'], color='lightgreen', axis=None)

display(styled_v)

# Save performance_before.csv (baseline only)
pd.DataFrame([metrics_baseline]).to_csv(PERFORMANCE_BEFORE, index=False)
print(f'✅ Saved: {PERFORMANCE_BEFORE}')

# Save performance_after.csv (all 4 methods)
df_long.to_csv(PERFORMANCE_AFTER, index=False)
print(f'✅ Saved: {PERFORMANCE_AFTER}')

# Print final clean table
print(f'\n{"="*75}')
print('  FINAL PERFORMANCE TABLE')
print(f'{"="*75}')
display(df_perf[['method','time_sec','cpu_percent','memory_mb',
                  'throughput_records_per_sec','speedup_vs_baseline']].style
        .format({
            'time_sec': '{:.4f}',
            'cpu_percent': '{:.2f}%',
            'memory_mb': '{:.2f}',
            'throughput_records_per_sec': '{:,.0f}',
            'speedup_vs_baseline': '{:.2f}x',
        })
        .highlight_min(subset=['time_sec','memory_mb'], color='lightgreen')
        .highlight_max(subset=['throughput_records_per_sec','speedup_vs_baseline'], color='lightgreen')
        .highlight_max(subset=['time_sec','memory_mb'], color='#ffcccc')
)

# Download all output files
from google.colab import files
for f in [PERFORMANCE_BEFORE, PERFORMANCE_AFTER, 'performance_comparison.png']:
    if os.path.exists(f):
        files.download(f)
        print(f'⬇️  Downloaded: {f}')

✅ Saved: benchmark_full_runs.csv

  VERIFIED FROM CSV — same table reloaded from benchmark_full_runs.csv



✅ Saved: performance_before.csv
✅ Saved: performance_after.csv

  FINAL PERFORMANCE TABLE


,method,time_sec,cpu_percent,memory_mb,throughput_records_per_sec,speedup_vs_baseline
0,pandas_baseline,0.9565,98.37%,37.83,"979,549",1.00x
1,pandas_optimized,0.9048,99.30%,0.90,"1,021,739",1.06x
2,polars,0.0552,97.97%,0.85,"17,602,830",17.33x
3,duckdb,0.1782,100.00%,0.13,"5,189,524",5.37x


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  Downloaded: performance_before.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  Downloaded: performance_after.csv
